# S&P 500 — Drift Adaptation on a Near-Unpredictable Stream

Applies the benchmark to daily S&P 500 direction (2005–2024), built in
`stock_market.ipynb`.

**Read the result before the plots: this stream has essentially no signal.**
Every strategy lands within noise of the majority-class rate. That is the
finding, and this notebook is written to make it visible rather than to hide it
behind a ranking table. See [docs/stock_market_notes.md](../docs/stock_market_notes.md).

In [ ]:
import sys; sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

import streams, pipeline
from experiments import run_grid, summarise, best_per_dataset, majority_rate

plt.rcParams["figure.figsize"] = (11, 4.5)

## 1. The stream, and two corrections to it

`streams.stock_sp500()` applies two fixes to the raw feature set:

**Scale-free features.** The raw `ma_5/20/50` and `macd` are price levels, and
the S&P rose ~4.8x over this window. A tree learns `ma_50 < 1200` — true in 2005,
never true after ~2013. That is price inflation masquerading as concept drift,
contaminating exactly what we measure. They are replaced with ratios.

**No infinities.** `volume_change` had one `inf` from a zero-volume prior day.

In [ ]:
samples, drift_points = streams.stock_sp500()
print(f"{len(samples):,} trading days, drift_points={drift_points} (unknown for real data)")
print(f"majority-class rate: {majority_rate(samples):.4f}")
print("\nfeatures (all scale-free):")
samples[0][0]

## 2. Label timing

The target for day *t* is whether day *t+1* closed higher. That label **does not
exist** at *t*. Standard prequential evaluation would do:

```python
y_pred = model.predict_one(x_t)
model.learn_one(x_t, y_t)     # y_t encodes t+1 — the future
```

`delay=1` holds each sample until its label would genuinely have arrived. The
cell below measures what the leak was worth.

In [ ]:
for label, delay in [("delay=0  (leaks tomorrow)", 0), ("delay=1  (correct)", 1)]:
    r = pipeline.run_strategy(samples, "baseline", "stock_sp500", (),
                              log_to_mlflow=False, delay=delay)
    print(f"{label:26s} acc={r['final_accuracy']:.4f}  "
          f"lift over majority={r['lift_over_majority']:+.4f}")

print("\nThe leak is worth almost nothing here — because there is no signal to leak.")
print("On a stream with real signal this gap would be large, which is why the")
print("check matters even when it comes back clean.")

## 3. Every strategy, tracked in MLflow

`run_grid` applies `delay=1` automatically for this stream (see
`experiments.STREAM_DELAY`) and logs every run to MLflow along with the
majority-class baseline, so a near-chance result cannot be mistaken for a real
one in the UI.

~1 min; ARF dominates the runtime.

In [ ]:
df = run_grid(datasets=["stock_sp500"], log_to_mlflow=True)
df[["strategy", "seed", "final_accuracy", "final_macro_f1",
    "majority_baseline", "runtime_sec"]].round(4)

### The result

`lift_over_majority` is the column that matters. Anything at or below zero means
the strategy has learned the class prior and nothing else.

In [ ]:
agg = summarise(df)
view = agg[["strategy", "acc_mean", "acc_std", "f1_mean",
            "majority_baseline", "lift_over_majority", "runtime_mean"]]
view.round(4)

In [ ]:
fig, ax = plt.subplots()
order = agg.sort_values("acc_mean")
ax.barh(order.strategy, order.acc_mean, color="#7f9fc4", alpha=.9)
ax.axvline(order.majority_baseline.iloc[0], color="crimson", ls="--", lw=2,
           label=f"always predict majority ({order.majority_baseline.iloc[0]:.4f})")
ax.set_xlim(0.50, 0.58)
ax.set_xlabel("mean accuracy")
ax.set_title("S&P 500 — no strategy meaningfully clears the majority-class line")
ax.legend(); ax.grid(alpha=.3, axis="x"); plt.show()

### Why macro-F1 is far below accuracy

A classifier that predicts one class for nearly every sample scores near the base
rate on accuracy but collapses on macro-F1, which averages per-class. The gap
between the two columns above is the tell.

In [ ]:
from collections import Counter, deque
from river import tree

m = tree.HoeffdingTreeClassifier()
preds, pending = Counter(), deque()
for x, y in samples:
    p = m.predict_one(x)
    if p is not None:
        preds[p] += 1
    pending.append((x, y))
    if len(pending) > 1:
        lx, ly = pending.popleft()
        m.learn_one(lx, ly)

total = sum(preds.values())
for k, v in sorted(preds.items()):
    print(f"predicted {k}: {v:5d}  ({v/total:.1%})")
print(f"\ntrue 'up' rate: {majority_rate(samples):.1%}")
print("The model predicts one class almost always. It has learned the prior.")

## 4. Where this sits against the other streams

The benchmark now spans five streams with real drift and one with none. That
range is what lets the paper say something about *when* adaptation is worth
deploying, rather than only which method wins.

In [ ]:
allruns = pd.read_csv("../results/raw_runs.csv")
allagg  = summarise(allruns)
best_per_dataset(allagg).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sub = allagg.dropna(subset=["lift_over_majority"])
piv = sub.pivot(index="dataset", columns="strategy", values="lift_over_majority")
piv.plot.bar(ax=ax, alpha=.9, rot=0)
ax.axhline(0, color="crimson", ls="--", lw=1.5)
ax.set_ylabel("accuracy − majority-class rate")
ax.set_title("Lift over the majority class: below zero is worse than guessing")
ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=8, ncol=3); plt.show()

## Conclusion

Daily index direction from technical indicators is close to unpredictable, and
the benchmark says so cleanly. **Do not report a strategy ranking on this
stream** — the differences are noise around 0.5445.

Three honest ways to use it:

1. **Report it as a negative result.** It extends the Airlines finding —
   adaptation machinery does not help without exploitable drift — into a domain
   where practitioners routinely assume otherwise. Quote the majority baseline
   beside every number.
2. **Change the target.** Volatility regime or large-move detection carries far
   more signal than direction, and has genuine regime shifts that would suit a
   drift benchmark.
3. **Drop the stream** and keep the five that produced usable results.

Option 2 is the strongest if the goal is a working drift benchmark; option 1 if
the goal is an honest paper from what already exists.

Every run is in MLflow: `mlflow ui --backend-store-uri sqlite:///mlflow.db`
